# Local vs Global Cost Functions in Deep Circuits


---

## Background

The plateau result has a useful consequence. It ties the gradient variance of a deep, 2-design circuit to a single property of the cost, the trace $\mathrm{Tr}(O^2)$ of the observable $O$ being measured, through $\mathrm{Var}[C] = \mathrm{Tr}(O^2)/[D(D+1)]$. The circuit itself drops out of the comparison, which means the choice of what to measure, rather than how the gates are arranged, becomes the handle on whether training can proceed.

Two costs make the point. A global cost asks the whole register to match a target bitstring at once, while a local cost adds up small checks on individual qubits and rewards each one on its own. The global projector $O_G = |0\rangle\langle 0|^{\otimes N} - I/D$ has a trace of order one, so its variance falls like $1/D^2 = 2^{-2N}$, the steepest plateau available. The normalized local observable $O_L = \frac{1}{N}\sum_j Z_j$ has a trace that grows with the Hilbert space, $\mathrm{Tr}(O_L^2) = D/N$, and its variance is $1/[N(D+1)]$, close to $1/(N\,2^N)$. Both still vanish, with the local exponent exactly half the global one.

Halving the exponent is not a small thing. The ratio $\mathrm{Var}_L/\mathrm{Var}_G \approx 2^N/N$ can be the difference between a circuit that still has a usable slope near twenty qubits and one whose slope vanished long before. The reason sits behind the arithmetic. A single-qubit term feels only the gates inside its backward light cone, the patch of circuit that can influence it, and when that patch is small the relevant Haar average is taken over a small unitary instead of the full $2^N$-dimensional one.

This is where the real escape lives. As long as the circuit stays shallow, the light cone of a local term covers only a handful of qubits, its effective dimension is fixed, and the gradient variance shrinks merely as $1/\mathrm{poly}(N)$ rather than exponentially. Let the depth grow until those light cones overlap and scramble the whole register, and the local cost loses its advantage and rejoins the exponential law computed here. Trainability of a local cost is therefore a statement about depth as much as about the observable.

## Motivation

Not all cost functions are equally trainable. A **global** cost (e.g., projecting onto the target state) and a **local** cost (e.g., measuring single-qubit observables) both suffer from barren plateaus in 2-design circuits, but the **rate** of exponential decay is dramatically different. The local cost has gradient variance $\sim 1/(N \cdot 2^N)$, while the global cost decays as $\sim 1/2^{2N}$, an exponential advantage for the local cost. This exercise derives both scalings via the same Weingarten calculus and quantifies the advantage.

## Preliminaries / Toolbox

**1. Cost Functions in VQAs:** The objective is $C(\boldsymbol{\theta}) = \langle 0| U^\dagger(\boldsymbol{\theta}) O U(\boldsymbol{\theta}) |0\rangle$.

**2. Global vs. Local Observables:** A global cost relies on an observable that acts non-trivially on all qubits, e.g., the traceless global projector $O_G = |0\rangle\langle 0|^{\otimes N} - I/D$. A local cost is a sum of local terms, e.g., $O_L = \frac{1}{N}\sum_j Z_j$.

**3. Barren Plateaus (2-design):** If $U(\boldsymbol{\theta})$ forms a 2-design, the variance of the cost function is $\mathrm{Var}[C] = \frac{\mathrm{Tr}(O^2)}{D(D+1)}$ where $D=2^N$.

**4. Trace properties:** $\mathrm{Tr}(A \otimes B) = \mathrm{Tr}(A)\mathrm{Tr}(B)$. The dimension is $D=2^N$.

## Exercise

For a 2-design circuit on $N$ qubits ($D = 2^N$), compare two traceless cost observables:

$$
O_G = |0\rangle\langle 0|^{\otimes N} - \frac{I}{D} \qquad \text{(global projector, traceless)},
$$

$$
O_L = \frac{1}{N}\sum_{j=1}^N Z_j \qquad \text{(normalized local observable, traceless)}.
$$

**(a)** Compute $\mathrm{Tr}(O_G^2)$ and $\mathrm{Tr}(O_L^2)$.

**(b)** Using the Weingarten formula, determine the gradient variance scaling for both costs.

**(c)** Compute the ratio $\mathrm{Var}_L / \mathrm{Var}_G$ and interpret.

## Solution

### Part (a): Computing $\mathrm{Tr}(O^2)$ for each cost

#### Global cost $O_G$:

$$
O_G^2 = \left(|0\rangle\langle 0|^{\otimes N} - \frac{I}{D}\right)^2 = |0\rangle\langle 0|^{\otimes N} - \frac{2}{D}|0\rangle\langle 0|^{\otimes N} + \frac{I}{D^2}.
$$

$$
\mathrm{Tr}(O_G^2) = 1 - \frac{2}{D} + \frac{1}{D} = 1 - \frac{1}{D} = \frac{D-1}{D}.
$$

For large $D$: $\mathrm{Tr}(O_G^2) \approx 1$.

#### Local cost $O_L$:

$$
O_L^2 = \frac{1}{N^2} \sum_{j,k} Z_j Z_k.
$$

- **Diagonal** ($j = k$): $\mathrm{Tr}(Z_j^2) = D$. There are $N$ such terms.
- **Off-diagonal** ($j \neq k$): $Z_j Z_k$ is a tensor product with two $Z$ factors and $N-2$ identities. Since $\mathrm{Tr}(Z) = 0$, $\mathrm{Tr}(Z_j Z_k) = 0$.

$$
\mathrm{Tr}(O_L^2) = \frac{1}{N^2} \cdot N \cdot D = \frac{D}{N}.
$$

### Part (b): Gradient variance via Weingarten calculus

From Exercise 7.1, we derived the exact Weingarten result for any traceless observable $O$:

$$
\mathbb{E}[C^2] = \frac{\mathrm{Tr}(O^2)}{D(D+1)}.
$$

(This followed from the $k=2$ Weingarten formula: the $\pi = \mathrm{id}$ contribution vanished because $\mathrm{Tr}(O) = 0$, and the $\pi = (12)$ contribution gave $\mathrm{Tr}(O^2) \times [\mathrm{Wg}(\mathrm{id}) + \mathrm{Wg}((12))] = \mathrm{Tr}(O^2)/(D(D+1))$.)

Applying this to both costs:

**Global:**

$$
\mathrm{Var}_G = \frac{(D-1)/D}{D(D+1)} = \frac{D-1}{D^2(D+1)} \approx \frac{1}{D^2} = 2^{-2N}.
$$

**Local:**

$$
\mathrm{Var}_L = \frac{D/N}{D(D+1)} = \frac{1}{N(D+1)} \approx \frac{1}{ND} = \frac{1}{N \cdot 2^N}.
$$

### Part (c): Ratio and interpretation

$$
\frac{\mathrm{Var}_L}{\mathrm{Var}_G} = \frac{D^2(D+1)}{D \cdot N \cdot (D+1)} \cdot \frac{D}{D-1} \approx \frac{D}{N} = \frac{2^N}{N}.
$$

$$
\boxed{\frac{\mathrm{Var}_L}{\mathrm{Var}_G} \approx \frac{2^N}{N} \gg 1.}
$$

The local cost has an **exponentially larger** gradient than the global cost.

**Physical interpretation:**

- The global cost $O_G = |0\rangle\langle 0|^{\otimes N}$ is a rank-1 projector, it asks "are you *exactly* in the target state?" Since $\mathrm{Tr}(O_G^2) \approx 1$ is $O(1)$, the Weingarten formula gives $\mathrm{Var}_G \sim 1/D^2$: the cost landscape is exponentially flat.

- The local cost $O_L = \frac{1}{N}\sum Z_j$ sums single-qubit measurements. Since $\mathrm{Tr}(O_L^2) \sim D/N$, the variance gets a factor $D/N$ boost. Each local term "sees" only part of the system, so the information is not diluted over the full $D$-dimensional Hilbert space.

- Both costs still have barren plateaus (exponential decay), but the exponent is halved for the local cost: $2^{-N}$ vs $2^{-2N}$. In practice, this can mean the difference between trainable ($N \sim 20$) and untrainable.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

D, N_sym = sp.symbols('D N', positive=True, integer=True)

# Traceless observables:  Tr(O_G^2) = (D-1)/D ~ O(1),  Tr(O_L^2) = D/N
# 2-design gradient variance ~ Tr(O^2)/(D(D+1)):
#   Global: Var_G ~ 1/(D(D+1))   ~ 2^(-2N)
#   Local:  Var_L = (D/N)/(D(D+1)) = 1/(N(D+1)) ~ 2^(-N)/N
print('Variance scaling (2-design circuits):')
print(f'  Global cost: Tr(O_G^2) ~ 1    ->  Var_G ~ 1/D^2   = 2^(-2N)')
print(f'  Local cost:  Tr(O_L^2) = D/N  ->  Var_L ~ 1/(N D) = 2^(-N)/N')
print()
for n in range(2, 10):
    d = 2**n
    var_g = 1/(d*(d+1))
    var_l = 1/(n*(d+1))
    print(f'  N={n}: Var_G={var_g:.2e}, Var_L={var_l:.2e}, ratio Var_L/Var_G={var_l/var_g:.1f}  (~D/N={d/n:.1f})')

---
## Numerical Verification

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

N_vals = np.arange(2, 13)
D_vals = 2.0**N_vals
var_global = (D_vals - 1) / (D_vals**2 * (D_vals + 1))   # ~ 2^{-2N}
var_local = 1.0 / (N_vals * (D_vals + 1))                 # ~ 1/(N 2^N)

plt.figure(figsize=(6, 4))
plt.semilogy(N_vals, var_local, 'o-', color='seagreen', label='Local cost')
plt.semilogy(N_vals, var_global, 's-', color='crimson', label='Global cost')
plt.xlabel('Number of qubits $N$')
plt.ylabel('Gradient variance')
plt.title('Local vs global cost: gradient variance vs $N$')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Takeaway

The global cost decays as $2^{-2N}$ and the local cost as $2^{-N}$, a separation of order $2^N/N$ that comes entirely from $\mathrm{Tr}(O^2)\sim D/N$ for the local observable against $O(1)$ for the global one. Locality halves the exponent of the plateau, so cost-function design sets the boundary between trainable and untrainable circuits at fixed depth.

In [ ]:
import numpy as np
from scipy.stats import unitary_group

np.random.seed(42)
sz = np.array([[1,0],[0,-1]], dtype=complex)

for N in [2, 3, 4, 5, 6]:
    D = 2**N
    psi0 = np.zeros(D, dtype=complex); psi0[0] = 1

    # Build observables
    O_G = np.outer(psi0, psi0) - np.eye(D)/D
    O_L = np.zeros((D, D), dtype=complex)
    for j in range(N):
        Oj = np.eye(1)
        for k in range(N):
            Oj = np.kron(Oj, sz if k == j else np.eye(2))
        O_L += Oj / N

    trOG2 = np.trace(O_G @ O_G).real
    trOL2 = np.trace(O_L @ O_L).real

    # Monte Carlo
    costs_G, costs_L = [], []
    for _ in range(8000):
        U = unitary_group.rvs(D)
        psi = U @ psi0
        costs_G.append((psi.conj() @ O_G @ psi).real)
        costs_L.append((psi.conj() @ O_L @ psi).real)

    var_G = np.var(costs_G)
    var_L = np.var(costs_L)
    pred_G = trOG2 / (D*(D+1))
    pred_L = trOL2 / (D*(D+1))

    print(f"N={N}: Tr(O_G²)={trOG2:.3f}, Tr(O_L²)={trOL2:.3f}")
    print(f"      Var_G={var_G:.2e} (pred {pred_G:.2e}), "
          f"Var_L={var_L:.2e} (pred {pred_L:.2e})")
    if var_G > 1e-10:
        print(f"      Var_L/Var_G = {var_L/var_G:.1f} (pred D/N = {D/N:.1f})")
    print()

print("Local cost has exponentially larger gradient.")

## Connections

This is the same barren-plateau second moment as in the [global-cost notebook](solution_ch7_barren_plateau.ipynb), now resolved by the locality of the observable. The underlying average is the Haar twirl introduced in the [Weingarten notebooks](../ch3/solution_ch3_weingarten_first_moment.ipynb). The contrast between the two cost functions shows that trainability is set not by the circuit alone but by how local the quantity you measure is.

## References

- Cerezo et al., *Cost function dependent barren plateaus in shallow parametrized quantum circuits*, Nature Communications (2021).
- Garcia et al., *Barren plateaus from learning scramblers with local cost functions*, Journal of High Energy Physics (2023).
- Larocca et al., *Barren plateaus in variational quantum computing*, Nature Reviews Physics (2025).
- Płodzień, *Information Scrambling, Quantum Chaos, and Haar-Random States*, lecture notes (2025). [arXiv:2511.14397](https://arxiv.org/abs/2511.14397)